# Homework 4 Solution

This notebook shows one possible solution for the multivariable linear regression homework based on the China panel dataset.

For readability, this solution creates a few short English aliases for Chinese variable names while keeping the original source columns intact.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "homework_04"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR

**Optional setup**

In [ ]:
%pip install -q openpyxl seaborn statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel(
    DATA_DIR / "china" / "china_panel_data.xlsx",
    sheet_name=0,
)

province_map = pd.read_csv(DATA_DIR / "china" / "province_name_mapping.csv")

df = df.merge(province_map, left_on="地区", right_on="region_cn", how="left")
df.head()

In [ ]:
df.columns

# Q1. Select regression variables and produce descriptive statistics.

A common choice is to use GDP per capita as the dependent variable and variables such as tertiary-industry value added, exports, e-commerce activity, and R&D expenditure as explanatory variables.

When the standard deviation is very large, a log transformation is often useful to reduce scale differences and potential heteroskedasticity.

In [ ]:
df["year"] = df["年份"]
df["gdp_per_capita"] = df["国内生产总值（当年价）（亿元）"] / df["总人口（万人）"]
df["lngdpc"] = np.log(df["gdp_per_capita"])
df["lnservice"] = np.log(df["第三产业增加值（亿元）"])
df["lnexp"] = np.log(df["货物出口金额（万美元）"])
df["ecommerce_share_pct"] = df["有电子商务交易活动企业占总企业数比重（%）"]
df["rd_expenditure"] = df["R&D经费（万元）"]
df["urban_employment"] = df["城镇就业人员数（万人）"]

df[[
    "gdp_per_capita",
    "lngdpc",
    "lnservice",
    "lnexp",
    "ecommerce_share_pct",
    "rd_expenditure",
    "urban_employment",
]].describe().T

# Q2. Visualize selected variables. Different regions should be visually distinguishable.

In [ ]:
plt.scatter(
    df["year"],
    df["rd_expenditure"],
    c="orange",
    alpha=0.5,
    label="R&D expenditure",
)
plt.scatter(
    df["year"],
    df["urban_employment"],
    c="pink",
    alpha=0.5,
    label="Urban employment",
)

plt.xlabel("Year")
plt.ylabel("Value")
plt.title("Two indicators over time")
plt.legend()
plt.savefig(MODULE_OUTPUT_DIR / "scatter_year_two_indicators.png", bbox_inches="tight")

In [ ]:
sns.scatterplot(
    data=df,
    x="year",
    y="ecommerce_share_pct",
    hue="region_en",
)
plt.legend(bbox_to_anchor=(1.0, 1.0), loc="upper left", ncol=2)
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "scatter_ecommerce_share_by_region.png", bbox_inches="tight")

In [ ]:
sns.scatterplot(data=df, x="region_en", y="lngdpc", hue="year")
plt.xlabel("Region")
plt.ylabel("Log GDP per capita")
plt.tick_params(
    direction="in",
    length=4,
    width=2,
    colors="black",
    grid_color="black",
    grid_alpha=0.1,
)
plt.legend(loc="best", ncol=1)
plt.xticks(rotation=45, ha="right")
plt.savefig(MODULE_OUTPUT_DIR / "scatter_lngdpc_by_region.png", bbox_inches="tight")

In [ ]:
df["area_group"] = "Non-coastal region"

df.loc[df["地区"].isin(["上海", "江苏", "浙江", "福建", "广东"]), "area_group"] = "Southeastern coastal region"

df["area_group"].unique()

In [ ]:
sns.scatterplot(data=df, x="year", y="lngdpc", hue="area_group", legend=True)
plt.legend(loc="best")
plt.savefig(MODULE_OUTPUT_DIR / "scatter_lngdpc_by_area_group.png", bbox_inches="tight")

# Q3. Run OLS regressions using the selected variables.

In [ ]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col

# Define the dependent variable.
y = df[["lngdpc"]]

# Define the independent variables.
X1 = df[["lnservice"]]
X2 = df[["lnservice", "lnexp"]]
X3 = df[["lnservice", "lnexp", "ecommerce_share_pct"]]
X4 = df[["lnexp", "ecommerce_share_pct"]]

# Add constant terms.
X1 = sm.add_constant(X1)
X2 = sm.add_constant(X2)
X3 = sm.add_constant(X3)
X4 = sm.add_constant(X4)

# Fit OLS models.
model1 = sm.OLS(y, X1).fit()
model2 = sm.OLS(y, X2).fit()
model3 = sm.OLS(y, X3).fit()
model4 = sm.OLS(y, X4).fit()

# Create a summary table with significance stars.
reg = summary_col(
    [model1, model2, model3, model4],
    stars=True,
    model_names=["Model 1", "Model 2", "Model 3", "Model 4"],
    float_format="%0.2f",
    info_dict={"N": lambda result: result.nobs},
)

reg_df = reg.tables[0].reset_index(drop=False)
reg_df.to_excel(MODULE_OUTPUT_DIR / "Homework4_regression_results.xlsx", index=False)

reg_df